In [2]:
# ============================================================
# FULL END-TO-END CODE (TRAIN + SAVE + PREDICT @ THRESHOLD 0.80)
# - Trains Stage A (Benign vs Attack) + Stage B (Attack Family)
# - Saves bundle to artifacts/ids_ensemble.joblib
# - Runs inference on Dataset/ and SAVES ONLY predicted attacks
#   using a FIXED decision threshold of 0.80
# Output files:
#   1) artifacts/ids_ensemble.joblib
#   2) predicted_attacks_thr_080.csv
# ============================================================

import os, glob, re, warnings, joblib, time, sys, itertools, threading, unicodedata
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve

warnings.filterwarnings("ignore")

# =========================
# CONFIG
# =========================
DATA_DIR      = "Dataset"
BENIGN_NAME   = "BENIGN"
RANDOM_STATE  = 42

FAST_MODE     = True      # True = fewer trees/iters, faster training
USE_STACKING  = True      # False = RF baseline only

# Stage-A threshold selection (training-time)
# If you want to still compute an "optimized" threshold, keep THR_MODE="recall".
# But your FINAL decision threshold requested is 0.80 at prediction time.
THR_MODE      = "recall"  # "fixed" or "recall"
TARGET_RECALL = 0.95      # used only if THR_MODE="recall"

# FINAL prediction threshold requested by you:
PRED_THR      = 0.80

# Save paths
BUNDLE_PATH   = "artifacts/ids_ensemble.joblib"
OUT_ATTACKS   = "predicted_attacks_thr_080.csv"

# =========================
# UTIL
# =========================
def step(msg, n=None, total=None):
    tag = f"[{n}/{total}] " if n and total else ""
    print(f"\n{tag}{msg}\n" + "-"*60)

def spinner(msg="Working..."):
    done_flag = {"done": False}
    def spin():
        for c in itertools.cycle('|/-\\'):
            if done_flag["done"]:
                break
            sys.stdout.write(f'\r{msg} {c}')
            sys.stdout.flush()
            time.sleep(0.1)
        sys.stdout.write('\r' + ' ' * (len(msg) + 4) + '\r')
    t = threading.Thread(target=spin)
    t.daemon = True
    t.start()
    return lambda: done_flag.__setitem__("done", True)

def normalize_colname(name):
    s = str(name).strip()
    s = s.replace("%", "pct")
    s = re.sub(r"[\/\\\s\-\.\(\)]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_").lower()
    return s

def clean_label(s: str) -> str:
    s = unicodedata.normalize("NFKC", str(s))
    s = s.replace("–", "-").replace("—", "-")
    s = re.sub(r"\s+", " ", s)
    return s.strip().upper()

def safe_numeric(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    out = out.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    # hardening: clip extreme values
    return out.clip(lower=-1e12, upper=1e12)

def load_all_csvs(data_dir: str) -> pd.DataFrame:
    files = sorted(glob.glob(os.path.join(data_dir, "*.csv")))
    if not files:
        raise SystemExit(f"No CSVs found in {data_dir}")
    dfs = []
    for f in files:
        d = pd.read_csv(f, low_memory=False)
        d.columns = [normalize_colname(c) for c in d.columns]
        d["capture_id"] = os.path.basename(f).replace(".csv", "")
        dfs.append(d)
    return pd.concat(dfs, ignore_index=True), files

# =========================
# [1] LOAD DATA
# =========================
step("Loading dataset CSVs", 1, 6)
df, files = load_all_csvs(DATA_DIR)
print("Files used:", len(files))
print("Combined shape:", df.shape)

# =========================
# [2] LABELS
# =========================
step("Creating labels", 2, 6)
label_col = next((c for c in df.columns if "label" in c.lower()), None)
if not label_col:
    raise SystemExit("Label column not found (must contain 'label').")

df[label_col]      = df[label_col].astype(str)
df["label_attack"] = df[label_col].map(clean_label)
df["label_binary"] = (df["label_attack"] != BENIGN_NAME).astype(int)

print("Top labels:")
print(df["label_attack"].value_counts().head(15))

# =========================
# [3] CLEAN NUMERICS + DROP IDENTIFIERS
# =========================
step("Cleaning numeric columns & dropping identifiers", 3, 6)
exclude = {"capture_id", label_col, "label_binary", "label_attack"}

# convert to numeric
for c in tqdm(df.columns, desc="Converting to numeric"):
    if c in exclude:
        continue
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.replace([np.inf, -np.inf], np.nan).fillna(0.0)

# drop potential identifiers / leak columns
drop_candidates = []
id_like_patterns = [
    "flow_id",
    "src_ip", "dst_ip", "source_ip", "destination_ip",
    "src_port", "dst_port",
    "timestamp",
    "simillarhttp"
]
for c in df.columns:
    if c in exclude:
        continue
    lc = c.lower()
    if any(p in lc for p in id_like_patterns):
        drop_candidates.append(c)

df = df[[c for c in df.columns if c not in drop_candidates]]
print("Dropped", len(drop_candidates), "identifier-like columns.")

# =========================
# [4] FEATURES + SPLIT
# =========================
step("Selecting features & splitting train/test", 4, 6)
features = [c for c in df.columns if c not in exclude and np.issubdtype(df[c].dtype, np.number)]
print("Feature count:", len(features))

X = df[features]
y_bin = df["label_binary"]
y_mul = df["label_attack"]

X_temp, X_test, y_temp_bin, y_test_bin, y_temp_mul, y_test_mul = train_test_split(
    X, y_bin, y_mul,
    test_size=0.3,
    stratify=y_bin,
    random_state=RANDOM_STATE
)

attack_mask = y_temp_bin == 1
X_train_atk = X_temp[attack_mask]
y_train_mul = y_temp_mul[attack_mask]

X_test_atk = X_test[y_test_bin == 1]
y_test_mul_atk = y_test_mul[y_test_bin == 1]

print(f"Stage A - Train: {X_temp.shape}, Test: {X_test.shape}")
print(f"Stage B - Train: {X_train_atk.shape}, Test: {X_test_atk.shape}")

# =========================
# [5] TRAIN STAGE A + STAGE B
# =========================
step("Training Stage A + Stage B", 5, 6)

# Preprocessor
pre = ColumnTransformer([("num", StandardScaler(with_mean=False), features)], remainder="drop")

# Base models
rf_trees = 150 if FAST_MODE else 350
gb_estim = 60  if FAST_MODE else 150

rf = RandomForestClassifier(
    n_estimators=rf_trees,
    n_jobs=-1,
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE
)

gb = GradientBoostingClassifier(
    n_estimators=gb_estim,
    learning_rate=0.1 if FAST_MODE else 0.05,
    random_state=RANDOM_STATE
)

try:
    from xgboost import XGBClassifier
    xgb = XGBClassifier(
        n_estimators=250 if FAST_MODE else 800,
        max_depth=6,
        learning_rate=0.05 if FAST_MODE else 0.03,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        tree_method="hist",
        eval_metric="logloss",
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbosity=0 if FAST_MODE else 1
    )
except Exception:
    xgb = gb

meta = LogisticRegression(
    max_iter=1500 if FAST_MODE else 2500,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

# Stage A classifier
if USE_STACKING:
    clfA = StackingClassifier(
        estimators=[("rf", rf), ("gb", gb), ("xgb", xgb)],
        final_estimator=meta,
        stack_method="predict_proba",
        n_jobs=-1,
        passthrough=False
    )
else:
    clfA = rf

model_bin = Pipeline([("pre", pre), ("clf", clfA)])

# Fit Stage A
print("\n⏳ Fitting Stage A (Benign vs Attack) ...")
stop = spinner("Fitting Stage A, please wait...")
t0 = time.time()
model_bin.fit(X_temp, y_temp_bin)
stop()
print(f"Elapsed time : {time.time()-t0:.2f} s")

# Choose a stored threshold (optional; NOT the final threshold you requested)
proba_test = model_bin.predict_proba(X_test)[:, 1]
if THR_MODE == "fixed":
    best_thr = 0.5
else:
    prec, rec, th = precision_recall_curve(y_test_bin, proba_test)
    idx = np.where(rec[:-1] >= TARGET_RECALL)[0]
    best_thr = float(th[idx[-1]]) if len(idx) > 0 else 0.5

predA = (proba_test >= best_thr).astype(int)
print("\n✅ Stage A evaluation (stored threshold, for reporting)")
print("Stored threshold:", best_thr)
print(classification_report(y_test_bin, predA, digits=4))
print("ROC-AUC:", roc_auc_score(y_test_bin, proba_test))
print("Confusion matrix:\n", confusion_matrix(y_test_bin, predA))

# Fit Stage B on attacks only
clfB_core = RandomForestClassifier(
    n_estimators=100 if FAST_MODE else 200,
    max_depth=20,
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE
)
model_mul = Pipeline([("pre", pre), ("clf", clfB_core)])

print("\n⏳ Fitting Stage B (Attack Family) ...")
stop = spinner("Fitting Stage B, please wait...")
t0 = time.time()
model_mul.fit(X_train_atk, y_train_mul)
stop()
print(f"Elapsed time : {time.time()-t0:.2f} s")

predB = model_mul.predict(X_test_atk)
print("\n✅ Stage B evaluation")
print(classification_report(y_test_mul_atk, predB, digits=4))

# Save bundle
os.makedirs("artifacts", exist_ok=True)
bundle = {
    "bin": model_bin,
    "mul": model_mul,
    "features": features,
    "best_threshold": float(best_thr),  # stored (optimized) threshold for reporting
    "trained_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "config": {
        "FAST_MODE": FAST_MODE,
        "USE_STACKING": USE_STACKING,
        "THR_MODE": THR_MODE,
        "TARGET_RECALL": TARGET_RECALL,
        "PRED_THR_REQUESTED": PRED_THR,  # your requested threshold
    }
}
joblib.dump(bundle, BUNDLE_PATH)
print("\n💾 Saved bundle to:", BUNDLE_PATH)

# =========================
# [6] INFERENCE ON DATASET (ATTACKS ONLY @ 0.80)
# =========================
step(f"Inference: save ONLY predicted attacks @ threshold={PRED_THR:.2f}", 6, 6)

# Reload bundle (ensures you test what you saved)
B = joblib.load(BUNDLE_PATH)
model_bin = B["bin"]
model_mul = B["mul"]
features  = B["features"]

# Re-load raw dataset for inference (keeps original label intact)
df_inf, _ = load_all_csvs(DATA_DIR)

# keep true labels for checking (optional)
label_col_inf = next((c for c in df_inf.columns if "label" in c.lower()), None)
if label_col_inf:
    df_inf["true_label"] = df_inf[label_col_inf].astype(str).map(clean_label)

# Align columns to training features
df_inf.columns = [normalize_colname(c) for c in df_inf.columns]
X_inf = df_inf.reindex(columns=features, fill_value=0.0)
X_inf = safe_numeric(X_inf)

# Stage A probabilities
df_inf["prob_attack"] = model_bin.predict_proba(X_inf)[:, 1]
df_inf["is_attack"] = df_inf["prob_attack"] >= PRED_THR

attacks = df_inf[df_inf["is_attack"]].copy()

print("\n===== PREDICTION SUMMARY =====")
print("Total flows:", len(df_inf))
print("Predicted attacks @0.80:", len(attacks))
print("Max prob_attack:", float(df_inf["prob_attack"].max()))
print("Mean prob_attack:", float(df_inf["prob_attack"].mean()))

# Stage B on attacks only
if len(attacks) > 0:
    X_atk = attacks.reindex(columns=features, fill_value=0.0)
    X_atk = safe_numeric(X_atk)

    attacks["predicted_family"] = model_mul.predict(X_atk)

    if hasattr(model_mul, "predict_proba"):
        fam_proba = model_mul.predict_proba(X_atk)
        attacks["family_confidence"] = fam_proba.max(axis=1)
    else:
        attacks["family_confidence"] = np.nan

# Save attacks
attacks.to_csv(OUT_ATTACKS, index=False)
print("\n✅ Saved predicted attacks to:", OUT_ATTACKS)

# Preview
cols = ["capture_id", "prob_attack", "predicted_family", "family_confidence"]
if "true_label" in attacks.columns:
    cols.insert(2, "true_label")
print("\nPreview (first 20 predicted attacks):")
print(attacks[cols].head(20))



[1/6] Loading dataset CSVs
------------------------------------------------------------
Files used: 8
Combined shape: (2830743, 80)

[2/6] Creating labels
------------------------------------------------------------
Top labels:
label_attack
BENIGN                        2273097
DOS HULK                       231073
PORTSCAN                       158930
DDOS                           128027
DOS GOLDENEYE                   10293
FTP-PATATOR                      7938
SSH-PATATOR                      5897
DOS SLOWLORIS                    5796
DOS SLOWHTTPTEST                 5499
BOT                              1966
WEB ATTACK � BRUTE FORCE         1507
WEB ATTACK � XSS                  652
INFILTRATION                       36
WEB ATTACK � SQL INJECTION         21
HEARTBLEED                         11
Name: count, dtype: int64

[3/6] Cleaning numeric columns & dropping identifiers
------------------------------------------------------------


Converting to numeric: 100%|███████████████████████████████████████████████████████████████████████| 82/82 [00:00<00:00, 130.16it/s]


Dropped 0 identifier-like columns.

[4/6] Selecting features & splitting train/test
------------------------------------------------------------
Feature count: 78
Stage A - Train: (1981520, 78), Test: (849223, 78)
Stage B - Train: (390352, 78), Test: (167294, 78)

[5/6] Training Stage A + Stage B
------------------------------------------------------------

⏳ Fitting Stage A (Benign vs Attack) ...
Fitting Stage A, please wait... |Elapsed time : 11291.65 s
                                   
✅ Stage A evaluation (stored threshold, for reporting)
Stored threshold: 0.999859849387428
              precision    recall  f1-score   support

           0     0.9880    0.9998    0.9939    681929
           1     0.9993    0.9504    0.9742    167294

    accuracy                         0.9901    849223
   macro avg     0.9936    0.9751    0.9840    849223
weighted avg     0.9902    0.9901    0.9900    849223

ROC-AUC: 0.999950720837223
Confusion matrix:
 [[681810    119]
 [  8299 158995]]

⏳ Fi